In [1]:
!pip install -q gdown transformers accelerate

import os, sys
from pathlib import Path
import gdown

# Dataset + helper code live in the shared IOAI-2026/AnimalDeduction/dataset folder
# (public link). Download locally and import from there — no sign-in needed.
LOCAL_DIR = Path('/content/animaldeduction')
if not LOCAL_DIR.exists() or not any(LOCAL_DIR.iterdir()):
    gdown.download_folder(id='1YheHvGfQw5YUa7MjdUF0hQC4sdtLZ5UC',
                          output=str(LOCAL_DIR), quiet=True, use_cookies=False)

sys.path.insert(0, str(LOCAL_DIR))
os.chdir(LOCAL_DIR)
print('Working directory:', os.getcwd())
print('Files:', sorted(p.name for p in LOCAL_DIR.iterdir()))

Working directory: /content/animaldeduction
Files: ['animals_pool.txt', 'dev.csv', 'evaluate.py', 'interactor.py', 'questions_pool.txt', 'test1.csv']


In [2]:
import random
import numpy as np
import pandas as pd
import torch

from interactor import Interactor
from evaluate import evaluate, load_pools

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

animals_pool, questions_pool = load_pools()
print(f'animals_pool size:   {len(animals_pool):>6}  (e.g. {animals_pool[:5]})')
print(f'questions_pool size: {len(questions_pool):>6}  (e.g. {questions_pool[:3]})')

# Sanity probe: create one Interactor and ask two questions about an octopus.
probe = Interactor(gold_animal='octopus', animals_pool=animals_pool, questions_pool=questions_pool)
print("\nask('is it a mammal?')      ->", probe.ask('is it a mammal?'))
print("ask('does it live in water?') ->", probe.ask('does it live in water?'))
print('Queries used:', probe.queries_used, '/', probe.budget)

# The model loads in bfloat16, which a T4 (Turing) cannot accelerate. Convert to
# float16 (T4 has fp16 tensor cores) -> identical answers, several times faster.
import torch
if torch.cuda.is_available() and next(Interactor._model.parameters()).dtype != torch.float16:
    Interactor._model = Interactor._model.half()
    print('oracle model -> float16 (faster on T4)')

Device: cpu
animals_pool size:     1472  (e.g. ['lion', 'tiger', 'leopard', 'snow leopard', 'cheetah'])
questions_pool size:    559  (e.g. ['is it a mammal?', 'is it a bird?', 'is it a reptile?'])


  [interactor] loading Qwen/Qwen2.5-3B-Instruct on cpu...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  [interactor] LLM ready.

ask('is it a mammal?')      -> no
ask('does it live in water?') -> yes
Queries used: 2 / 15


In [3]:
class RandomBaseline:
    def __init__(self, animals_pool, questions_pool, seed=0):
        self.animals_pool = animals_pool
        self.questions_pool = questions_pool
        self.rng = random.Random(seed)

    def solve(self, interactor):
        guessed = set()
        while not interactor.is_done():
            cand = self.rng.choice(self.animals_pool)
            while cand in guessed:
                cand = self.rng.choice(self.animals_pool)
            guessed.add(cand)
            interactor.guess(cand)

baseline_results = evaluate(RandomBaseline(animals_pool, questions_pool), 'dev.csv')

  25/150 rows  mean_score=0.0000  (0.0s)
  50/150 rows  mean_score=0.0156  (0.0s)
  75/150 rows  mean_score=0.0219  (0.0s)
  100/150 rows  mean_score=0.0242  (0.0s)
  125/150 rows  mean_score=0.0194  (0.1s)
  150/150 rows  mean_score=0.0161  (0.1s)

  Dataset:       dev.csv
  Mean score:    0.0161
  Solved rate:   2.0%
  Mean queries:  14.89 / 15
  Wall time:     0.1s


In [4]:
# Reference solution (optional, slow to init). Demonstrates precompute + match,
# but uses FIXED, non-adaptive questions -> leaves most of the score on the table.
FIXED_QUESTIONS = [
    'is it a mammal?',
    'is it a bird?',
    'is it a fish?',
    'is it an insect?',
    'does it live in water?',
    'can it fly?',
    'is it a carnivore?',
    'is it bigger than a human?',
    'does it have a backbone?',
    'is it commonly kept as a pet?',
    'does it have legs?',
    'does it lay eggs?',
]

class FixedQuestionsReference:
    def __init__(self, animals_pool, questions_pool, max_animals=None):
        self.animals_pool = animals_pool
        self.questions_pool = set(questions_pool)
        self.fixed = [q for q in FIXED_QUESTIONS if q in self.questions_pool]
        from interactor import Interactor
        cand = animals_pool if max_animals is None else animals_pool[:max_animals]
        self.candidates = cand
        print(f'  [reference] precomputing {len(cand)} x {len(self.fixed)} answer table...')
        self.table = {}
        for i, a in enumerate(cand):
            sim = Interactor(gold_animal=a, animals_pool=self.animals_pool,
                             questions_pool=self.questions_pool, budget=10**9)
            self.table[a] = tuple(1 if sim.ask(q) == 'yes' else 0 for q in self.fixed)
            if (i + 1) % 200 == 0:
                print(f'    {i+1}/{len(cand)}')
        print('  [reference] table ready.')

    def solve(self, interactor):
        obs = []
        for q in self.fixed:
            if interactor.remaining_budget() <= 1:
                break
            obs.append(1 if interactor.ask(q) == 'yes' else 0)
        obs = tuple(obs)
        def agree(a):
            vec = self.table[a]
            return sum(1 for x, y in zip(vec, obs) if x == y)
        ranked = sorted(self.candidates, key=agree, reverse=True)
        for a in ranked:
            if interactor.is_done():
                break
            interactor.guess(a)

# Example (commented out by default — uncomment to run; slow to init):
# ref = FixedQuestionsReference(animals_pool, questions_pool)
# ref_results = evaluate(ref, 'dev.csv')

In [5]:
class MySolution:
    def __init__(self, animals_pool, questions_pool):
        self.animals_pool = animals_pool
        self.questions_pool = questions_pool
        # TODO: precompute the (animal x question) answer table and anything else.

    def solve(self, interactor):
        # TODO: your strategy. Each ask is ~1 bit; log2(1400) ~ 10.5 bits needed.
        # Budget is 15. Choose each question to split the remaining candidates.
        while not interactor.is_done():
            interactor.guess(self.animals_pool[0])

my_dev = evaluate(MySolution(animals_pool, questions_pool), 'dev.csv')

  25/150 rows  mean_score=0.0000  (0.0s)
  50/150 rows  mean_score=0.0000  (0.0s)
  75/150 rows  mean_score=0.0000  (0.0s)
  100/150 rows  mean_score=0.0000  (0.0s)
  125/150 rows  mean_score=0.0000  (0.1s)
  150/150 rows  mean_score=0.0000  (0.1s)

  Dataset:       dev.csv
  Mean score:    0.0000
  Solved rate:   0.0%
  Mean queries:  15.00 / 15
  Wall time:     0.1s


In [6]:
import os

solution = MySolution(animals_pool, questions_pool)
dev_results   = evaluate(solution, 'dev.csv')
test1_results = evaluate(solution, 'test1.csv')

splits = [('dev', dev_results), ('test1', test1_results)]
# test2 is a held-out set; included automatically only if present in the folder.
if os.path.exists('test2.csv'):
    splits.append(('test2', evaluate(solution, 'test2.csv')))

rows = [{
    'split': name, 'n': r['n'], 'mean_score': r['mean_score'],
    'solved_rate': r['solved_rate'], 'mean_queries': r['mean_queries'],
} for name, r in splits]

# FINAL = n-weighted mean over every test split available (test1 [+ test2]).
tests = [r for name, r in splits if name.startswith('test')]
n_test = sum(r['n'] for r in tests)
rows.append({
    'split': 'FINAL',
    'n': n_test,
    'mean_score':   sum(r['mean_score']   * r['n'] for r in tests) / n_test,
    'solved_rate':  sum(r['solved_rate']  * r['n'] for r in tests) / n_test,
    'mean_queries': sum(r['mean_queries'] * r['n'] for r in tests) / n_test,
})
pd.DataFrame(rows)

  25/150 rows  mean_score=0.0000  (0.0s)
  50/150 rows  mean_score=0.0000  (0.0s)
  75/150 rows  mean_score=0.0000  (0.0s)
  100/150 rows  mean_score=0.0000  (0.1s)
  125/150 rows  mean_score=0.0000  (0.1s)
  150/150 rows  mean_score=0.0000  (0.1s)

  Dataset:       dev.csv
  Mean score:    0.0000
  Solved rate:   0.0%
  Mean queries:  15.00 / 15
  Wall time:     0.1s
  25/500 rows  mean_score=0.0000  (0.0s)
  50/500 rows  mean_score=0.0000  (0.0s)
  75/500 rows  mean_score=0.0000  (0.0s)
  100/500 rows  mean_score=0.0000  (0.0s)
  125/500 rows  mean_score=0.0000  (0.1s)
  150/500 rows  mean_score=0.0000  (0.1s)
  175/500 rows  mean_score=0.0000  (0.1s)
  200/500 rows  mean_score=0.0000  (0.1s)
  225/500 rows  mean_score=0.0000  (0.1s)
  250/500 rows  mean_score=0.0000  (0.1s)
  275/500 rows  mean_score=0.0000  (0.1s)
  300/500 rows  mean_score=0.0000  (0.1s)
  325/500 rows  mean_score=0.0000  (0.2s)
  350/500 rows  mean_score=0.0000  (0.2s)
  375/500 rows  mean_score=0.0000  (0.2s)
  

,split,n,mean_score,solved_rate,mean_queries
0,dev,150,0.0,0.0,15.0
1,test1,500,0.0,0.0,15.0
2,FINAL,500,0.0,0.0,15.0
